###Real-time scoring simulation

In [ ]:
#Simulate real-time transaction scoring

# Select random transactions from test set
n_transactions = 10000
sample_indices = np.random.choice(len(X_test_scaled), n_transactions, replace=False)
X_sample = X_test_scaled[sample_indices]
y_sample = y_test.iloc[sample_indices]

best_model = models[best_model_name]

In [ ]:
# Real-time predictions
risk_scores = best_model.predict_proba(X_sample)[:, 1]

In [ ]:
# Apply thresholds
decisions = pd.DataFrame({
    'transaction_id': range(len(sample_indices)),
    'actual_fraud': y_sample.values,
    'risk_score': risk_scores,
    'conservative_flag': risk_scores >= risk_thresholds['conservative'],
    'optimal_flag': risk_scores >= risk_thresholds['optimal'],
    'aggressive_flag': risk_scores >= risk_thresholds['aggressive'],
    'amount': df.iloc[sample_indices]['Amount'].values
})

In [ ]:
# Priority scoring (Amount × Risk Score)
decisions['priority_score'] = decisions['amount'] * decisions['risk_score']
decisions = decisions.sort_values('priority_score', ascending=False)

In [ ]:
# Summary statistics
print(f"• High-risk transactions (conservative): {decisions['conservative_flag'].sum()}")
print(f"• High-risk transactions (optimal): {decisions['optimal_flag'].sum()}")
print(f"• High-risk transactions (aggressive): {decisions['aggressive_flag'].sum()}")
print(f"• Actual frauds in sample: {decisions['actual_fraud'].sum()}")

• High-risk transactions (conservative): 39
• High-risk transactions (optimal): 28
• High-risk transactions (aggressive): 22
• Actual frauds in sample: 20


In [ ]:
# Alert prioritization
alerts = decisions[decisions['optimal_flag'] == True].copy()
if len(alerts) > 0:
  print(f"• Average risk score of alerts: {alerts['risk_score'].mean():.3f}")
  print(f"• Average amount of alerts: ${alerts['amount'].mean():.2f}")
  print(f"• Top priority alert amount: ${alerts['amount'].iloc[0]:.2f}")

simulation_results = decisions

• Average risk score of alerts: 0.960
• Average amount of alerts: $579.94
• Top priority alert amount: $12910.93
